# Check out SNOW-17 products

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import seaborn as sns
import copy

from pathlib import PurePath

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')

# Set seaborn palette
sns.set_palette('icefire')

In [ ]:
%reload_ext autoreload

In [ ]:
data_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/CBRFC_SNOW17'

# CBRFC zone data

In [ ]:
csv_fns = h.fn_list(data_dir, '*.csv')
print(f'Found {len(csv_fns)} CSV files in {data_dir}')
_ = [print(PurePath(csv).stem) for csv in csv_fns]

In [ ]:
# To prepare the filtered dataset if not yet complete

# Locate and read in the corresponding polygons for these codes
poly_fn = h.fn_list(data_dir, 'polygons/*.shp')[0]
print(poly_fn)

# Read this in
polys = gpd.read_file(poly_fn)

# filter by the codes in the CSV files, looks like this is segment
codes = [PurePath(csv).stem for csv in csv_fns]
polys = polys[polys['segment'].isin(codes)]


# Write this to file, dropping the index
out_fn = f'{data_dir}/CBRFC_SNOW17_select_segments.gpkg'
if not os.path.exists(out_fn):
    polys.to_file(out_fn, driver='GPKG', index=False)
    print(f'Wrote {out_fn}')
else:
    # Read it back in
    polys = gpd.read_file(out_fn)

polys.head()

### Compute zonal means for iSnobal data with calc_cbrfc_zonalmean.py

In [ ]:
import calc_cbrfc_zonalmean as cczm

In [ ]:
basin = 'yampa'
WY = 2021
workdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/model_runs/'
basindirs, basin_polys = cczm.prep_basin_data(basin=basin, WY=WY, data_dir=data_dir, workdir=workdir)

In [ ]:
# day_list = cczm.load_isnobal_data(basindirs)

In [ ]:
# Reproject polygon to EPSG 32613
basin_polys = basin_polys.to_crs('EPSG:32613')
topo, zones, zone_names_sorted = cczm.locate_topo_zones(basin, basin_polys)
print(zones, len(zones), len(zone_names_sorted))

In [ ]:
var = 'specific_mass'

In [ ]:
ds = day_list[0]
runtype = 'baseline'
zonal_gdf = gpd.GeoDataFrame(columns=['zone', 'zonal_classification', 'geometry'])
zonal_gdf['zone'] = basin_polys['zone']
zonal_gdf['geometry'] = basin_polys['geometry']

In [ ]:
zone_names_sorted

In [ ]:
# Work backwards from the existing csvs and reduce the zonal_gdf
subzones = np.unique(subzones)
basin_csv_fns = []
for zone in subzones:
    zone_csv = h.fn_list(data_dir, f'{zone}*csv')[0]
    print(zone_csv)
    basin_csv_fns.append(zone_csv)
basin_csv_fns

In [ ]:
zonal_gdf.crs = 'EPSG:32613'
zonal_gdf.to_crs('EPSG:4326', inplace=True)
zonal_gdf

In [ ]:
import hvplot.pandas

In [ ]:
# Also plot the zonal gdf zones with hvplot with hover
zonal_gdf.hvplot(
    geo=True,
    tiles='OSM',
    hover_cols=['zone'],
    title=f'{basin} {runtype} zones',
    width=800,
    height=600,
    alpha=0.5,
    line_color='black'
# ).opts(
#     xaxis=None,
#     yaxis=None
).opts(
    toolbar='above'
).opts(
    frame_width=800
).opts(
    frame_height=600
)

In [ ]:
topo.rio.write_crs('EPSG:32613', inplace=True)
topo_4326 = topo.rio.reproject('EPSG:4326', inplace=True)
# topo_4326

In [ ]:
# Plot zonal_gdf on top of topo_4326
zonal_gdf.hvplot(
    geo=True,
    tiles='OSM',
    hover_cols=['zone'],
    title=f'{basin} {runtype} zones',
    width=800,
    height=600,
    alpha=0.5,
    line_color='black'
).opts(
    toolbar='above'
).opts(
    frame_width=800
).opts(
    frame_height=600
) * topo_4326['cbrfc_zone'].hvplot(
    geo=True,
    tiles='OSM',
    title=f'{basin} {runtype} topo zones',
    width=800,
    height=600,
    alpha=0.2,
    cmap='Grays',
    ).opts(
        frame_width=800
    ).opts(
        frame_height=600
    )

In [ ]:
# Set equal aspect ratio and plot the topo cbrfc zones with viridis colormap
topo_4326['cbrfc_zone'].hvplot(
    geo=True,
    tiles='OSM',
    title=f'{basin} {runtype} topo zones',
    width=800,
    height=600,
    alpha=0.8,
    cmap='rainbow',
    ).opts(
        frame_width=800
    ).opts(
        frame_height=600
    )

In [ ]:
# Assign zonal_classification by unique zone values
zonal_gdf['zonal_classification'] = zones
zonal_gdf

In [ ]:

# Reduce the dataset to only the variables we care about
ds_var = ds[var].compute()
# Loop through each day
for jdx, day in enumerate(ds_var.time):
    # Grab date in YYYYMMDD format
    date_str = day.dt.strftime('%Y%m%d').values
    print(date_str)
    date_list = []
    # Loop through each zone
    for zone in zones:
        # Calculate the daily spatial mean for this zone
        daily_mean = float(ds_var.isel(time=jdx).where(topo[zone_name] == zone, drop=True).mean())
        # Append the mean thickness to this date column list
        date_list.append(daily_mean)
    # Concatenate this day's list as a new column in the zonal_gdf by first converting to dataframe
    zonal_gdf = pd.concat([zonal_gdf, pd.DataFrame({f'{runtype}_{date_str}': date_list})], axis=1)

# # Save the zonal_gdf to a geopackage
# if save_file is not None:
#     out_fn = f'{data_dir}/isnobal_zonal/CBRFC_SNOW17_{basin}_WY{WY}_{runtype}_zonal_{var}_gdf.gpkg'
#     zonal_gdf.to_file(out_fn, driver='GPKG', index=False)
#     print(f'Wrote {out_fn}')

# zonal_gdf = cczm.build_cbrfc_var_gdf(basin_polys=basin_polys, ds=ds, topo=topo,
#                                 basin=basin, WY=WY,
#                                 runtype=runtype, zones=zones,
#                                 data_dir=data_dir, var=var, zone_name='cbrfc_zone',
#                                 save_file=True)
# zonal_gdf_list.append(zonal_gdf)

In [ ]:
# # Archive this, may be useful in future, but TBD
# # Do this with rasters and all the data variables! This took about 5 minutes for one month in the Animas...
# # compared to 45 sec with the single variable geodataframe

# def calculate_daily_spatial_mean_by_zone(ds, topo, zone, zone_name='cbrfc_zone'):
#     """
#     Calculate the daily spatial mean for each variable in the dataset based on zone filtering and return a DataSet

#     Parameters:
#     ds (xarray.Dataset): Input dataset with time, x, and y dimensions.

#     Returns:
#     xarray.Dataset: Dataset with daily spatial means.
#     """
#     # Step 1: Filter dataset by zone
#     filtered_ds = ds.where(topo[zone_name] == zone, drop=True)
#     # Step 2: Calculate the spatial mean for each day
#     # Group by the date and calculate the mean over x and y dimensions
#     daily_spatial_mean = filtered_ds.groupby("time.date").mean(dim=["x", "y", "time"], skipna=True)
#     daily_spatial_mean.compute()

#     # Assign the value from each timestep of the daily_spatial_mean_expanded to all zonal values in corresponding timestep of result ds
#     result_ds = ds.copy()

#     for var in result_ds.data_vars:
#         for time_step in daily_spatial_mean['date']:
#             # Get the value for the current time step
#             value = daily_spatial_mean[var].sel(date=time_step).values
#             # Assign this value to all zonal values in the corresponding timestep of result_ds
#             result_ds[var] = result_ds[var].where(topo[zone_name]!=zone, value)
#     return result_ds

### iSnobal zonal data

In [ ]:
# Read in basin data
basin = 'animas'
WY = 2022
var = 'specific_mass'
zonal_basin_data_fns = h.fn_list(data_dir, f'isnobal_zonal/*{basin}*WY{WY}*{var}*gpkg')
zonal_basin_data_list = [gpd.read_file(zonal_fn) for zonal_fn in zonal_basin_data_fns]
# Convert to meters rather than equivalent to mm SWE for all columns that are not zone, zonal_classification or geometry
for zonal_basin_data in zonal_basin_data_list:
    for col in zonal_basin_data.columns:
        if col not in ['zone', 'zonal_classification', 'geometry']:
            zonal_basin_data[col] = zonal_basin_data[col] / 1000.0  # Convert from mm to m

In [ ]:
# Baseline
zonal_basin_data_list[0].head()

In [ ]:
# HRRR-SPIReS
zonal_basin_data_list[1].head()

In [ ]:
# Generate a difference gdf first by copying the zones and geometry
diff_gdf = zonal_basin_data_list[0][['zone', 'geometry']].copy()

# Calculate the difference for each zone based on the two dataframes and the date in column names
for col in zonal_basin_data_list[0].columns:
    if col not in ['zone', 'zonal_classification', 'geometry']:
        # Strip out the date from the column
        col_dt = col.split('_')[-1]
        # print(col_dt)
        # Check if the date exists as part of a column in both dataframes
        baseline_col = [ col for col in zonal_basin_data_list[0].columns if col_dt in col ]
        hrrr_spires_col = [ col for col in zonal_basin_data_list[1].columns if col_dt in col ]
        if len(baseline_col) > 0 and len(hrrr_spires_col) > 0:
            # print('yes')
            # Calculate the difference, convert to dataframe, and concat it to the diff_gdf
            diff_col = zonal_basin_data_list[1][hrrr_spires_col].values - zonal_basin_data_list[0][baseline_col].values
            diff_col = pd.DataFrame(diff_col, columns=[col_dt])
            diff_gdf = pd.concat([diff_gdf, diff_col], axis=1)
diff_gdf

In [ ]:
fig, axa = plt.subplots(1, 3, figsize=(12,6), sharex=True, sharey=True)
vmin, vmax = 0, 0.2
diff_vmin, diff_vmax = -0.02, 0.02
zonal_basin_data_list[0].plot(column=f'baseline_{WY-1}1231', ax=axa[0], vmin=vmin, vmax=vmax, legend=True, legend_kwds={'label': 'Baseline', 'fraction': 0.06})
zonal_basin_data_list[1].plot(column=f'hrrrspires_{WY-1}1231', ax=axa[1], vmin=vmin, vmax=vmax, legend=True, legend_kwds={'label': 'HRRR-SPIReS', 'fraction': 0.06})
diff_gdf.plot(column=f'{WY-1}1231', ax=axa[2], vmin=diff_vmin, vmax=diff_vmax, cmap='RdYlBu',
              legend=True, legend_kwds={'label': 'Difference (HRRR-SPIReS - Baseline)', 'fraction': 0.06})
for ax in axa.flatten():
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()

In [ ]:
test_isnobal_gdf = copy.deepcopy(zonal_basin_data_list[0])
# Set the crs
test_isnobal_gdf.crs = 'EPSG:32613'
# Convert to lat lon
test_isnobal_gdf.to_crs('EPSG:4326', inplace=True)
test_isnobal_gdf.hvplot(geo=True, c='baseline_20211103')

### Zone nomenclature

In [ ]:
# Grab the zone names from these basins
zones = np.unique(zonal_basin_data_list[0]['zone'].values)
# Trim the last three characters to match the zone names in the csv files
zones_abbrev = np.unique([zone[:-3] for zone in zones])
print(zones)
print(zones_abbrev)

### SNOW17 data


In [ ]:
# Read in SNOW17 data csvs only if csv name contains one of the zone names
basin_csv_fns = []
for zone in zones_abbrev:
    zone_csv = h.fn_list(data_dir, f'{zone}*csv')[0]
    basin_csv_fns.append(zone_csv)

df_list = [pd.read_csv(csv, index_col=0) for csv in basin_csv_fns]
for df in df_list:
    print(df.shape)

In [ ]:
def process_cbrfc_df(in_df, WY):
    '''
    Generate date column and filter to specified water year (WY)
    in_df: pandas DataFrame with columns 'cal_yr', 'mon', and 'zday'
    WY: Water year to filter the DataFrame to, e.g., 2022 for WY 2022
    '''
    df = copy.deepcopy(in_df)
    # Generate a date column from the cal_yr, mon and zday columns by using a lambda function to do row-wise conversion
    df['date'] = pd.to_datetime(df.apply(lambda row: f"{row['cal_yr']}-{row['mon']:02d}-{row['zday']:02d}", axis=1))

    # Drop the cal_yr, mon, and zday columns
    df.drop(columns=['cal_yr', 'mon', 'zday'], inplace=True)
    # Trim to the specified water year Oct 1 through September 30
    df = df[(df['date'] >= f'{WY - 1}-10-01') & (df['date'] <= f'{WY}-09-30')]
    return df
# Process each df in the list
wy_list = [process_cbrfc_df(df, WY) for df in df_list]

In [ ]:
# Now recompile the snow17 wy list dataframes into a single one by moving the opid and appending to the swe column
snow17_wy = pd.DataFrame()

# For each unique "subzone" in opid column, split into new df
# These should remain in zone order as above since deriving
# zone names from np.unique
snow17_wy_list = []
for df in wy_list:
    subzones = np.unique(df['opid'])
    for subzone in subzones:
        print(subzone)
        sub_df = df[df['opid'] == subzone]
        # Append subzone name to the swe column
        sub_df = sub_df.rename(columns={'swe': f'{subzone}_swe'})
        # Drop opid column
        sub_df.drop(columns=['opid'], inplace=True)
        print(sub_df.shape)
        # Convert date column to index
        sub_df.set_index('date', inplace=True)
        # Concatenate to the larger snow17_wy DataFrame
        snow17_wy = pd.concat([snow17_wy, sub_df], axis=1)
snow17_wy

In [ ]:
# Convert from inches to meters
snow17_wy = snow17_wy / 39.37
snow17_wy

In [ ]:
zonal_basin_data_list[0]

In [ ]:
# Turn into something compatible with hvplot
snow17_wy_hv = snow17_wy.reset_index()#.melt(id_vars='date', var_name='zone', value_name='swe')
snow17_wy_hv

In [ ]:
snow17_wy_hv.hvplot.line(x='date', y=snow17_wy_hv.columns[1:], width=800, height=400, title=f'SNOW17 SWE for {basin} WY{WY}', ylabel='SWE (m)', xlabel='Date',
                         line_width=2, alpha=0.8).opts(legend_position='top_right')

# Comparison with SNOW17 data

In [ ]:
def process_isnobal_zone_df(zone_row, verbose=False):
    # convert to series, dropping geometry, zonal classification
    zone_row = zone_row.drop(columns=['geometry', 'zonal_classification'])
    # Move the prepending model run to the zone column
    model_run = np.unique([c.split('_')[0] for c in zone_row.columns if c not in ['zone']])[0]
    if verbose:
        print(model_run)
    # Rename the columns to remove the prepending model run
    zone_row.rename(columns={c: c.split('_')[1] for c in zone_row.columns if '_' in c}, inplace=True)
    # Rename zone
    zone_row.rename(columns={'zone': f'zone_{model_run}_run'}, inplace=True)
    # Transpose zone_row
    zone_row = zone_row.T
    # Pull the zone name from the first row
    colname = zone_row.iloc[0]
    # Drop the first row
    zone_row = zone_row.drop(index=colname.name)
    # Rename column to zone name
    zone_row.columns = [colname]
    # Set zone_baseline_run as the index and convert to datetime
    zone_row.index = pd.to_datetime(zone_row.index)
    return zone_row


In [ ]:
# Now compare the data for each zone
# For each zone
nrows, ncols = 4, 2
fig, axa = plt.subplots(nrows, ncols, figsize=(ncols*5, nrows*3), sharex=True, sharey=True)
for ax, zone in zip(axa.flatten(), zones):
    # Plot time series of: Baseline, HRRR-SPIReS
    for df, label in zip(zonal_basin_data_list, ['Baseline', 'HRRR-SPIReS']):
        # Get the row for the zone
        zone_row = df[df['zone'] == zone]
        # Process the zone_row to get it in the right format
        zone_row = process_isnobal_zone_df(zone_row)
        # Plot the zone_row
        ax.plot(zone_row, label=label)
    # Plot CBRFC snow-17 data
    snow17_wy[f'{zone}_swe'].plot(ax=ax, label='CBRFC SNOW-17', color='black', linestyle='--')
    # TODO: On the second subplot, plot the difference from CBRFC
    # ax.set_title(f'Zone: {zone}')
    ax.set_ylabel('SWE (m)')
    ax.annotate(zone, xy=(0.96, 0.84), xycoords='axes fraction', ha='right', fontsize=11, fontstyle='italic')
    # if this is the first plot, add a legend
    if ax == axa[0, 0]:
        ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()

plt.suptitle(f'{basin} WY{WY} iSnobal comparison with CBRFC SNOW-17', fontsize=12, y=1)

savefig = False
fig_fn = f'{data_dir}/{basin}_wy{WY}_{var}_snow17comp.png'
print(fig_fn)
if savefig:
    fig.savefig(fig_fn, dpi=300, bbox_inches='tight')
plt.tight_layout()

In [ ]:
# Plot an interactive version of the above
import ipywidgets as widgets
from ipywidgets import interact

def plot_zone_swe(zone,
                  ):
    """
    Plot the SWE for a given zone
    """
    _, ax = plt.subplots(figsize=(10, 5))

    # Plot Baseline and HRRR-SPIReS data
    for df, label in zip(zonal_basin_data_list, ['Baseline', 'HRRR-SPIReS']):
        zone_row = df[df['zone'] == zone]
        zone_row = process_isnobal_zone_df(zone_row)
        ax.plot(zone_row.index, zone_row[zone], label=label)

    # Plot CBRFC SNOW-17 data
    ax.plot(snow17_wy.index, snow17_wy[f'{zone}_swe'], label='CBRFC SNOW-17', color='black', linestyle='--')
    ax.set_title(f'Zone: {zone}')
    ax.set_ylabel('SWE (m)')
    ax.legend()

    plt.show()

# Create a zone dropdown
zone_dropdown = widgets.Dropdown(
    options=zones,
    value=zones[0],
    description='Zone:'
)
# Create an interactive plot
interact(plot_zone_swe, zone=zone_dropdown,
         snow17_wy=snow17_wy, hover=True)

 # TODO:  Compute zonal statistics for each polygon using NWM and UA outputs